# OpenAI Agents SDK에서 Migration하기

OpenAI Agents SDK로 만든 app을 Claude Agent SDK로 port하려 한다면, 이 Notebook은 비용 승인 agent라는 하나의 예제로 각 primitive를 대응시킵니다.

**Migration 후 얻는 것.** Claude Agent SDK는 Claude Code와 같은 runtime에서 실행됩니다. 내장 `Read`, `Edit`, `Bash`, `Grep` tool, agent의 접근 대상을 제한하는 계층형 permission system, 자동 prompt caching, 진행 상황 streaming이나 실행 도중 개입을 위한 event stream 직접 접근을 그대로 활용할 수 있습니다. Tool 정의는 명시적이며(type-hint introspection에 의존하지 않고 schema를 선언), loop는 직접 구동합니다. 대부분의 port는 tool마다 boilerplate가 늘어나고 나머지 부분에서는 줄어듭니다.

## 이 Notebook을 마치면 다음을 할 수 있습니다:

- Business logic을 다시 작성하지 않고 `@function_tool`, guardrail, `Runner.run`을 Claude의 대응 기능으로 교체하기
- 단일 agent app port하기: custom tool, input/output guardrail, multi-turn session, 영속적인 resume
- `ClaudeSDKClient`와 stateless `query()` function을 사용할 때 구별하기
- SDK의 OpenTelemetry export를 기존 observability stack에 연결하기

두 SDK 모두 실제로 실행합니다. `.env`에 `OPENAI_API_KEY`와 `ANTHROPIC_API_KEY`가 필요합니다.

| OpenAI Agents SDK | Claude Agent SDK |
|---|---|
| `Agent(name, instructions, tools)` | `ClaudeAgentOptions` + system prompt |
| `@function_tool` | `@tool` + `create_sdk_mcp_server` |
| `@input_guardrail` | Loop 전의 일반 function(또는 `UserPromptSubmit` hook) |
| `@output_guardrail` | `ResultMessage.result`에 적용하는 일반 function |
| `Runner.run(agent, msg)` | `ClaudeSDKClient` context manager |
| `Sessions`(client 관리) | 같은 `ClaudeSDKClient` 재사용 |
| `conversation_id`(server 관리) | `resume=session_id`(disk 기반) |
| 내장 tracing dashboard | OTel-native — 기존 Grafana/Datadog/Honeycomb에 연결 |
| `handoffs=[...]` | `AgentDefinition` + Agent tool — 부록 참고 |

## 예제: 비용 승인

비용 제출을 승인하거나 검토 대상으로 표시하는 단일 agent입니다. Tool 하나(`check_policy`), input guardrail 하나(달러 금액이 없으면 거부), 그리고 loop로 구성됩니다.

먼저 OpenAI Agents SDK로 한 번 구축한 뒤 각 primitive를 port합니다.

## 사전 요구사항

**필수 지식**

- Python의 `async`/`await`에 익숙함
- Migration 대상인 OpenAI Agents SDK primitive에 익숙함

**필수 Tool**

- Python 3.11 또는 3.12
- `ANTHROPIC_API_KEY`([발급받기](https://console.anthropic.com)) 및 `OPENAI_API_KEY`([발급받기](https://platform.openai.com/api-keys)) — 두 SDK 모두 실제로 실행합니다

**권장 사항:**

- `query()` 및 `ClaudeSDKClient` 기초를 위해 [Notebook 00](./00_The_one_liner_research_agent.ipynb)을 훑어보세요

> `openai-agents`는 `0.9.3`으로 고정되어 있습니다. 아직 1.0 이전이며 API가 자주 바뀝니다. Version을 올리면 `Agent`/`Runner`/`@function_tool` signature를 다시 확인하세요.

In [1]:
%%capture
# openai-agents는 0.9.3으로 고정 — version을 올리기 전에 사전 요구사항 참고
%pip install "openai-agents==0.9.3" claude-agent-sdk python-dotenv

In [2]:
import os
import re
from dataclasses import replace

from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing from .env"
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY missing from .env"

OAI_MODEL = "gpt-4.1"
CLAUDE_MODEL = "claude-sonnet-4-6"

In [3]:
# ===== OpenAI Agents SDK — 전체 구현 =====
from agents import Agent, GuardrailFunctionOutput, Runner, function_tool, input_guardrail


@function_tool
def check_policy(category: str, amount: float) -> dict:
    """Category의 비용 정책을 조회합니다. 승인 한도를 반환합니다. 유효한 category: meals, travel, software, other."""
    limits = {"meals": 75.0, "travel": 500.0, "software": 200.0, "other": 50.0}
    return {
        "category": category,
        "limit": limits.get(category.lower(), 50.0),
        "requires_receipt": amount > 25.0,
    }


@input_guardrail
def has_dollar_amount(ctx, agent, user_input: str) -> GuardrailFunctionOutput:
    has_amount = bool(re.search(r"\$\d+", user_input))
    return GuardrailFunctionOutput(
        output_info={"has_amount": has_amount},
        tripwire_triggered=not has_amount,
    )


expense_agent = Agent(
    name="Expense Approver",
    instructions=(
        "You approve or flag expense submissions. "
        "Always call check_policy first to get the limit for the expense category. "
        "Approve if the amount is under the limit; otherwise flag for manager review."
    ),
    tools=[check_policy],
    input_guardrails=[has_dollar_amount],
    model=OAI_MODEL,
)


async def run_oai(msg: str) -> str:
    result = await Runner.run(expense_agent, msg)
    return result.final_output


print(await run_oai("Lunch with Acme, $47"))

This lunch expense of $47 falls under the "meals" category, which has an approval limit of $75. Since the amount is below the limit, I approve the expense (assuming a receipt is provided, as required by policy).


## Primitive별 Port하기

### `@function_tool` → `@tool` + `create_sdk_mcp_server`

**OpenAI:** `@function_tool`은 type hint 및 docstring에서 tool schema를 도출합니다. Decorator가 적용된 function은 바로 `tools=[...]`에 들어갑니다.

**Claude:** `@tool`은 name, description, schema를 명시적인 argument로 받으며, 작성한 내용 그대로 model에 전달됩니다. Handler는 `async`이며 하나의 `args` dict를 받아 `{"content": [{"type": "text", "text": ...}]}`를 반환합니다. Tool은 in-process MCP server로 묶입니다(이름과 달리 subprocess나 network transport는 없음). 이것이 agent에 전달하는 단위입니다.

Function 내부 로직은 바뀌지 않습니다. Wrapper만 바뀝니다.

In [4]:
import json

from claude_agent_sdk import create_sdk_mcp_server, tool


@tool(
    "check_policy",
    "Look up the expense policy for a category. Returns the approval limit. Valid categories: meals, travel, software, other.",
    {"category": str, "amount": float},
)
async def check_policy_claude(args):
    limits = {"meals": 75.0, "travel": 500.0, "software": 200.0, "other": 50.0}
    category, amount = args["category"], args["amount"]
    result = {
        "category": category,
        "limit": limits.get(category.lower(), 50.0),
        "requires_receipt": amount > 25.0,
    }
    return {"content": [{"type": "text", "text": json.dumps(result)}]}


policy_server = create_sdk_mcp_server(name="expense", tools=[check_policy_claude])

### `Agent(...)` → `ClaudeAgentOptions`

`ClaudeAgentOptions`는 실제로 Claude에 전송되는 구성인 model, tool, system prompt를 담습니다. Guardrail 로직은 framework 객체에 등록하는 대신 application 코드에 남습니다. OpenAI의 `Agent`는 이들을 모두 함께 묶습니다.

| `Agent(...)` field | `ClaudeAgentOptions` 대응 항목 |
|---|---|
| `instructions=...` | `system_prompt=...` |
| `tools=[fn]` | `mcp_servers={...}` + `allowed_tools=[...]` |
| `model=...` | `model=...` |
| `name=...` | 필요 없음 — options는 이름을 가진 entity가 아닌 구성 값입니다. Subagent 이름은 `AgentDefinition`에 지정합니다(부록 참고). |
| `input_guardrails=[...]` | 실행 function에서 호출 — 다음 절 참고 |

`allowed_tools`의 tool 이름은 `mcp__{server_name}__{tool_name}` pattern을 따릅니다.

> **Permission에 대한 참고:** `allowed_tools`는 allow-rule이며 agent가 tool을 *사용할 수 있도록* 합니다. 사용자 승인 없이 호출할 수 있는지는 `permission_mode`에 달려 있습니다. `check_policy` 같은 읽기 전용 custom tool은 기본적으로 자유롭게 실행됩니다. 파일을 쓰거나 shell command를 실행하는 tool은 `permission_mode="acceptEdits"` 또는 `"bypassPermissions"`를 설정하지 않으면 승인을 요청합니다. Tool이 읽기 이외의 작업을 한다면 [permission guide](https://platform.claude.com/docs/en/agent-sdk/permissions)를 참고하세요.

In [5]:
from claude_agent_sdk import ClaudeAgentOptions

expense_system_prompt = (
    "You approve or flag expense submissions. "
    "Always call check_policy first to get the limit for the expense category. "
    "Approve if the amount is under the limit; otherwise flag for manager review."
)

expense_options = ClaudeAgentOptions(
    model=CLAUDE_MODEL,
    system_prompt=expense_system_prompt,
    mcp_servers={"expense": policy_server},
    allowed_tools=["mcp__expense__check_policy"],
)

> **내장 Tool.** 비용 agent는 custom tool만 사용하지만 SDK는 `Read`, `Edit`, `Bash`, `Grep` 등을 제공합니다. 이름으로 `allowed_tools`에 추가하세요(예: `allowed_tools=["Read", "Grep", "mcp__expense__check_policy"]`). Filesystem pattern은 [Notebook 00](./00_The_one_liner_research_agent.ipynb)을 참고하세요.

### `@input_guardrail` / `@output_guardrail` → 호출 전/후 검사

**OpenAI:** `@input_guardrail`은 사용자의 message를, `@output_guardrail`은 agent의 최종 답변을 검증합니다. 둘 다 차단 시 `GuardrailFunctionOutput(tripwire_triggered=True)`를 반환하며, 이로 인해 `InputGuardrailTripwireTriggered`(또는 output version)가 발생합니다. Input guardrail은 agent와 동시에 실행됩니다.

**Claude:** 검사는 일반 function입니다. Client를 시작하기 전에 input 검사를 호출하고, loop 종료 후 `ResultMessage.result`에 output 검사를 호출합니다.

| OAI | Claude |
|---|---|
| `@input_guardrail` decorator | Loop 전에 호출하는 일반 function |
| `@output_guardrail` decorator | Loop 이후 `result.result`에 호출하는 일반 function |
| `tripwire_triggered=True`가 예외 발생 | 일찍 반환(input) 또는 결과 override(output) |
| `Agent(input_guardrails=[...])`에 등록 | 실행 function에서 호출 |

#### 대안인 `UserPromptSubmit` Hook

SDK의 `UserPromptSubmit` hook은 prompt가 Claude에 도달하기 전에 발생하며 이를 차단할 수 있습니다. 구조적으로 `@input_guardrail`에 가장 가깝습니다. 아래에서 둘 다 보여 주지만 두 가지 이유로 일반 function을 사용해 시연합니다. 거부 message를 직접 제어할 수 있고(hook이 차단하면 `ResultMessage.result`가 비어 반환되며 이유는 호출자에게 전달되지 않음), guardrail 로직이 options 객체에 등록되는 대신 실행 function에 드러납니다. `@output_guardrail`에 명확히 대응하는 hook은 없습니다. `Stop`은 응답 완료 후 발생하지만 출력을 다시 쓰지는 않습니다. Tool 호출을 제한하는 `PreToolUse` hook은 다른 사용 사례이며 [Notebook 03](./03_The_site_reliability_agent.ipynb)을 참고하세요.

In [6]:
def has_dollar_amount_check(user_input: str) -> tuple[bool, str | None]:
    """Input 검사 — (allowed, rejection_message)를 반환합니다."""
    if re.search(r"\$\d+", user_input):
        return True, None
    return False, "I need a dollar amount to process this. Please include one (e.g., '$47')."


def has_decision_check(result: str) -> tuple[bool, str | None]:
    """Output 검사 — (allowed, override_message)를 반환합니다."""
    # Stem match는 demo를 위해 의도적으로 관대하게 설정했으며 production에서는 강화하세요.
    if re.search(r"\b(approv|flag|review)", result, re.IGNORECASE):
        return True, None
    return False, "I couldn't reach a clear approve/flag decision. Please resubmit."

비교를 위해 같은 검사를 `UserPromptSubmit` hook으로 연결한 예제입니다. Callback은 `input_data["prompt"]`를 받고, 허용하려면 `{}`, 차단하려면 `{"decision": "block", "reason": "..."}`를 반환합니다. 여기서는 `UserPromptSubmit`이 모든 prompt에서 발생하고 filtering할 tool 이름이 없으므로 `HookMatcher`는 `matcher` argument를 받지 않습니다.

시험하려면 다음 절의 `ClaudeSDKClient(...)` 호출에서 `options=expense_options`를 `options=hooked_options`로 교체하세요. 한 가지 주의점은 `UserPromptSubmit` hook이 차단할 때 현재 차단 이유가 `ResultMessage.result`에 전달되지 않는다는 것입니다. 거부 text 대신 빈 문자열이 반환됩니다. Demo loop가 반환을 직접 제어하는 일반 function을 사용하는 이유입니다.

사용자가 *보내는 것* 대신 agent가 *하는 것*을 제한하는 tool 수준 guardrail은 [Notebook 03](./03_The_site_reliability_agent.ipynb)의 `PreToolUse` hook pattern을 참고하세요.

In [7]:
from claude_agent_sdk import HookMatcher


async def has_dollar_amount_hook(input_data, tool_use_id, context):
    # UserPromptSubmit의 tool_use_id는 None — 모든 hook type에서 통일된 signature입니다.
    if re.search(r"\$\d+", input_data["prompt"]):
        return {}
    return {"decision": "block", "reason": "I need a dollar amount to process this."}


hooked_options = ClaudeAgentOptions(
    model=CLAUDE_MODEL,
    system_prompt=expense_system_prompt,
    mcp_servers={"expense": policy_server},
    allowed_tools=["mcp__expense__check_policy"],
    hooks={"UserPromptSubmit": [HookMatcher(hooks=[has_dollar_amount_hook])]},
)

### `Runner.run()` → `ClaudeSDKClient`

**OpenAI:** `result = await Runner.run(agent, msg)`가 loop를 실행하고 result 객체를 반환합니다. Text는 `result.final_output`에서 읽습니다.

**Claude:** `ClaudeSDKClient`는 async context manager입니다. `.query(msg)`로 전송한 뒤 `.receive_response()`를 iterate하여 도착하는 event를 받습니다. 모든 tool 호출, text block, 최종 결과를 순서대로 봅니다. Event type은 다음과 같습니다:

| Event | 포함 내용 |
|---|---|
| `SystemMessage` | Session 초기화 metadata |
| `AssistantMessage` | Text block 또는 tool-use block |
| `UserMessage` | Tool-result block |
| `ResultMessage` | 항상 마지막: `.result`, `.usage`, `.total_cost_usd` |

최종 답변은 `messages[-1].result`입니다.

> SDK는 내장 tool을 사용하는 일회성 호출용 stateless `query()` function도 제공합니다([Notebook 00](./00_The_one_liner_research_agent.ipynb) 참고). `create_sdk_mcp_server`로 custom tool을 제공할 때는 `ClaudeSDKClient`가 올바른 선택입니다. 영속적인 transport가 in-process MCP handshake를 처리하기 때문입니다.

In [8]:
from utils.agent_visualizer import display_agent_response, print_activity, reset_activity_context

from claude_agent_sdk import ClaudeSDKClient


async def run_claude(msg: str) -> None:
    # Input guardrail — agent 실행 전에 short-circuit
    allowed, rejection = has_dollar_amount_check(msg)
    if not allowed:
        print(rejection)
        return

    reset_activity_context()
    messages = []
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query(msg)
        async for event in client.receive_response():
            print_activity(event)
            messages.append(event)

    # receive_response()는 ResultMessage가 마지막으로 yield되는 event임을 보장함
    final = messages[-1].result
    ok, override = has_decision_check(final or "")
    if not ok:
        print(override)
        return

    display_agent_response(messages)


await run_claude("Lunch with Acme, $47")

🤖 Thinking...
🤖 Thinking...
🤖 Using: mcp__expense__check_policy()
✓ Tool completed
🤖 Thinking...


### Session → `ClaudeSDKClient`(in-memory) 또는 `resume=`(disk 기반)

OpenAI는 두 가지 session mode를 제공합니다:

| OAI mode | 영속성 | Claude 대응 방식 |
|---|---|---|
| `result.to_input_list()` + 재전송 | Client가 memory에 기록 보유 | 하나의 `ClaudeSDKClient` 재사용 — 열린 동일 context에서 `.query()`를 다시 호출 |
| `conversation_id` | Server-side, process 재시작 후에도 유지 | `resume=session_id` — 대화 transcript를 로컬 disk에 작성하여 재시작 후에도 유지 |

**In-memory(아래):** `ClaudeSDKClient` context를 열어 두고 `.query()`를 다시 호출합니다. 기록을 다시 보내지 않습니다. Process가 종료되면 사라집니다.

**Disk 기반:** 모든 실행은 `session_id`를 생성합니다(`ResultMessage.session_id`). 이를 저장한 뒤 다음 실행에 `ClaudeAgentOptions(resume=session_id, ...)`를 전달하세요. Transcript는 Anthropic server가 아닌 로컬 filesystem에 있습니다. 재시작 후에는 유지되지만 컴퓨터 간에는 유지되지 않습니다.

어느 mode에서든 대화가 context window보다 커지면 [`../misc/session_memory_compaction.ipynb`](../misc/session_memory_compaction.ipynb)을 참고하세요.

In [9]:
# --- In-memory: 같은 client 재사용 ---
async def run_claude_multiturn():
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query("Lunch with Acme, $47")
        async for _ in client.receive_response():
            pass  # Stream 소비

        # 같은 client — turn 1을 기억함
        await client.query("What about $90?")
        turn2 = [m async for m in client.receive_response()]
        # receive_response()는 ResultMessage가 마지막으로 yield되는 event임을 보장함
        return turn2[-1].result


print(await run_claude_multiturn())

🚩 **Flagged for Manager Review**

| Detail | Info |
|---|---|
| **Description** | Lunch with Acme |
| **Category** | Meals |
| **Amount** | $90.00 |
| **Policy Limit** | $75.00 |
| **Over Limit By** | $15.00 |
| **Status** | **Flagged** |

The $90 lunch exceeds the $75 meals limit by **$15.00**, so it requires manager approval before it can be reimbursed.


In [10]:
# --- Disk 기반: session_id 수집 후 나중에 resume ---

# Turn 1: ResultMessage에서 session_id 수집(항상 마지막 event)
async with ClaudeSDKClient(options=expense_options) as client:
    await client.query("Lunch with Acme, $47")
    turn1 = [m async for m in client.receive_response()]
session_id = turn1[-1].session_id

# Turn 2: 새 client, 새 process — disk에서 resume
resume_opts = replace(expense_options, resume=session_id)
async with ClaudeSDKClient(options=resume_opts) as client:
    await client.query("What about $90?")
    turn2 = [m async for m in client.receive_response()]
print(f"[resumed {session_id[:8]}...] {turn2[-1].result}")

[resumed 4fc6dc1e...] Here's the rundown for both expenses:

| Expense | Amount | Limit | Status |
|---|---|---|---|
| Lunch with Acme | $47 | $75 | ✅ **Approved** |
| Lunch with Acme | $90 | $75 | 🚩 **Flagged for Manager Review** |

- **$47** – Under the $75 meals limit, so it's **approved**. (Receipt required.)
- **$90** – Exceeds the $75 meals limit by $15, so it's **flagged for manager review**.


## 나란히 비교하기

세 test 입력을 두 SDK 모두에 실행합니다. 둘 다 $47 점심은 승인하고, $650 항공편은 검토 대상으로 표시하며, 달러 금액이 없는 입력은 거부할 것으로 예상합니다.

In [11]:
from agents.exceptions import InputGuardrailTripwireTriggered


async def run_claude_quiet(msg: str) -> str:
    # run_claude와 같은 guardrail — 최소 비교 helper를 위해 여기서 중복 구현
    allowed, rejection = has_dollar_amount_check(msg)
    if not allowed:
        return rejection
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query(msg)
        messages = [m async for m in client.receive_response()]
        # receive_response()는 ResultMessage가 마지막으로 yield되는 event임을 보장함
        return messages[-1].result or ""


test_inputs = [
    ("Lunch with Acme, $47", "approve"),
    ("Flight to NYC, $650", "flag"),
    ("Need approval for the thing", "guardrail"),
]

for msg, expected in test_inputs:
    print(f"\n{'=' * 60}\nINPUT: {msg}  (expect: {expected})\n{'=' * 60}")

    try:
        oai_out = await run_oai(msg)
    except InputGuardrailTripwireTriggered:
        oai_out = "[guardrail: InputGuardrailTripwireTriggered]"

    claude_out = await run_claude_quiet(msg)

    print(f"OAI:    {str(oai_out)[:200]}")
    print(f"Claude: {str(claude_out)[:200]}")


INPUT: Lunch with Acme, $47  (expect: approve)
OAI:    The expense for lunch ($47) is under the meals category limit of $75. This expense is approved.
Claude: ✅ **Expense Approved**

| Detail | Info |
|---|---|
| **Description** | Lunch with Acme |
| **Category** | Meals |
| **Amount** | $47.00 |
| **Meal Limit** | $75.00 |
| **Status** | **Approved** |

Yo

INPUT: Flight to NYC, $650  (expect: flag)
OAI:    The approval limit for travel expenses is $500. Since your flight to NYC costs $650, this expense exceeds the limit and will be flagged for manager review.
Claude: 🚩 **Expense Flagged for Manager Review**

| Detail | Info |
|---|---|
| **Description** | Flight to NYC |
| **Category** | Travel |
| **Amount** | $650.00 |
| **Policy Limit** | $500.00 |
| **Over Lim

INPUT: Need approval for the thing  (expect: guardrail)
OAI:    [guardrail: InputGuardrailTripwireTriggered]
Claude: I need a dollar amount to process this. Please include one (e.g., '$47').


## Tracing

Claude는 팀에서 이미 사용하는 standard로 출력합니다.

**실행별 Metrics는 이미 제공됩니다.** 모든 `ResultMessage`에는 `.total_cost_usd`와 `.usage`(input/output token, cache read, cache write, turn 수)가 있습니다. 아래 caching cell에서 확인할 수 있습니다. Programmatic 비용 추적은 [`../observability/usage_cost_api.ipynb`](../observability/usage_cost_api.ipynb)을 참고하세요.

**OpenTelemetry는 기본 지원됩니다.** SDK는 Claude Code runtime을 공유하며, 활성화하면 OTel metrics와 event를 export합니다:

```bash
export CLAUDE_CODE_ENABLE_TELEMETRY=1
export OTEL_METRICS_EXPORTER=otlp
export OTEL_LOGS_EXPORTER=otlp
export OTEL_EXPORTER_OTLP_PROTOCOL=grpc
export OTEL_EXPORTER_OTLP_ENDPOINT=http://your-collector:4317
```

Port에 이 설정은 필수가 아닙니다. Telemetry가 켜져 있든 꺼져 있든 agent는 동일하게 실행됩니다.

모든 tool 호출과 API 요청은 `prompt.id` tag가 있는 event를 내보내므로 모든 activity를 원래 prompt와 연결할 수 있습니다. 기존 Grafana / Datadog / Honeycomb / Langfuse stack을 대상으로 지정하세요. 전체 event schema는 [monitoring 문서](https://code.claude.com/docs/en/monitoring-usage)를 참고하세요.

## Claude 전용: Prompt Caching

System prompt 및 tool schema는 첫 호출 후 자동으로 cache되며, 구성이 필요하지 않습니다. 같은 prefix를 가진 이후 호출은 해당 prefix에 대해 input token 비용의 약 10%를 지불합니다.

> 이전 cell에서 이미 `expense_options`로 실행했으므로, 여기 도달할 때는 cache가 warm 상태입니다. 따라서 Run 1에서도 상당한 `cache_read_input_tokens`가 나타납니다. Cold 상태의 Notebook에서는 Run 1은 주로 `cache_creation`, Run 2는 주로 `cache_read`일 것입니다. Tool-calling loop는 query마다 여러 API 요청을 보내며 각각 cached prefix를 확장합니다. 따라서 반복 실행해도 cache_creation은 0이 되지 않지만, cache_read가 증가하면서 줄어들어야 합니다.

In [12]:
for i in range(2):
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query("Coffee, $6")
        messages = [m async for m in client.receive_response()]
    # receive_response()는 ResultMessage가 마지막으로 yield되는 event임을 보장함
    usage = messages[-1].usage or {}
    print(
        f"Run {i + 1}: cache_creation={usage.get('cache_creation_input_tokens', 0):>5}  "
        f"cache_read={usage.get('cache_read_input_tokens', 0):>5}  "
        f"input={usage.get('input_tokens', 0):>5}"
    )

Run 1: cache_creation=38728  cache_read=66272  input=    4
Run 2: cache_creation=  659  cache_read=95025  input=    4


---
## 부록: `handoffs` Migration하기

OpenAI Agents SDK app이 `handoffs=[...]`를 사용한다면, 여기서 사고방식이 달라집니다.

**OpenAI:** `handoffs=[specialist]`는 `transfer_to_specialist` tool을 노출합니다. 호출되면 specialist가 대화를 **넘겨받습니다**. 첫 agent는 종료됩니다.

**Claude:** Subagent는 `AgentDefinition`으로 programmatic하게 정의하고 `ClaudeAgentOptions(agents={...})`에 전달합니다. Orchestrator는 Agent tool로 **위임**하고 결과를 돌려받으며 제어권을 유지합니다.

단순 router(triage → specialist)에서는 차이가 작습니다. "Agent B가 넘겨받고 A는 다시 실행되지 않는다"는 동작이 핵심인 app에서는 LLM이 결정하는 routing 대신 간단한 Python dispatcher를 고려하세요.

> Claude Code CLI workflow에서 `.claude/agents/*.md` 파일로 정의한 subagent를 볼 수도 있습니다(예: [Notebook 01](./01_The_chief_of_staff_agent.ipynb)). 이는 대화형 사용을 위한 filesystem 기반 방식입니다. 코드로 제공하는 SDK application에는 아래의 programmatic `agents=` parameter가 자연스럽게 맞으며 filesystem dependency가 없습니다.

In [13]:
from claude_agent_sdk import AgentDefinition

approver = AgentDefinition(
    description="Approves expenses under policy limits. Use for any submission where the amount is at or under the limit returned by check_policy.",
    prompt="You approve expense submissions that are within policy. Confirm the amount and category, and remind the submitter if a receipt is required.",
    tools=["mcp__expense__check_policy"],
)

escalator = AgentDefinition(
    description="Escalates over-limit expenses to a manager. Use when the submitted amount exceeds the policy limit for its category.",
    prompt="You escalate over-limit expenses. Draft a one-line note to the manager: include the amount, the category, and how far over the limit it is.",
    tools=["mcp__expense__check_policy"],
)

triage_options = ClaudeAgentOptions(
    model=CLAUDE_MODEL,
    system_prompt="Route each expense to the appropriate subagent. Delegate to approver if under limit, escalator if over.",
    mcp_servers={"expense": policy_server},
    allowed_tools=["Agent", "mcp__expense__check_policy"],
    agents={"approver": approver, "escalator": escalator},
)

print(f"Orchestrator configured with subagents: {list(triage_options.agents)}")

# 구성만 정의하며 여기서는 실행하지 않습니다. 전체 multi-agent 실행은
# ./01_The_chief_of_staff_agent.ipynb을 참고하세요

Orchestrator configured with subagents: ['approver', 'escalator']


## 결론

### 구축한 것

Primitive별로 port한 비용 승인 agent 하나입니다. `@function_tool` → `@tool`, `Agent` → `ClaudeAgentOptions`, `@input_guardrail`/`@output_guardrail` → 일반 호출 전/후 검사, `Runner.run` → `ClaudeSDKClient`, 그리고 in-memory 및 disk 기반 session입니다.

### 핵심 사항

- **Tool:** 명시적인 schema는 작성한 내용 그대로 model에 전달됨을 뜻하며, introspection으로 인한 예상 밖의 결과가 없습니다. In-process MCP server로 묶어 `ClaudeSDKClient`로 전달하세요.
- **Guardrail:** 일반 function은 거부 흐름을 framework가 아닌 자신의 코드에 둡니다. Options에 등록하고 싶다면 `UserPromptSubmit` hook을 사용할 수 있습니다.
- **Session:** In-memory multi-turn에는 하나의 `ClaudeSDKClient`를 사용합니다. 재시작 후에도 유지되는 disk 기반 대화에는 `resume=session_id`를 사용합니다.
- **Tracing:** OpenTelemetry-native이므로 이미 운영하는 stack에 연결할 수 있습니다. Turn별 비용 및 token 사용량은 모든 `ResultMessage`에 있으므로 추가 연결이 필요하지 않습니다.

### 다음 단계

- Multi-agent orchestration: [01_The_chief_of_staff_agent.ipynb](./01_The_chief_of_staff_agent.ipynb)
- Tool-lifecycle hook(`PreToolUse`/`PostToolUse`): [03_The_site_reliability_agent.ipynb](./03_The_site_reliability_agent.ipynb)
- 비용 및 사용량 추적 pattern: [../observability/usage_cost_api.ipynb](../observability/usage_cost_api.ipynb)